In [5]:
# Cell 1: Load data and find credit transactions
import json
import re
import random
import shutil
from pathlib import Path

PROJECT = Path.home() / "llm-mail-trainer"

# Load all emails
with open(PROJECT / "data/parsed/emails.json", 'r') as f:
    all_emails = json.load(f)

print(f"✅ Loaded {len(all_emails):,} emails")

# Find credit transactions
credit_emails = []

for email in all_emails:
    body = email['body'].lower()
    subject = email['subject'].lower()
    combined = f"{subject} {body}"
    
    # Must have "credited" and amount
    if 'credited' in combined:
        if re.search(r'(?:rs\.?|₹)\s*[\d,]+', combined):
            credit_emails.append(email)

print(f"Credit transaction emails found: {len(credit_emails)}")

# Show samples
print("\n=== SAMPLE CREDIT EMAILS ===")
for i, email in enumerate(credit_emails[:5]):
    print(f"\n{i+1}. Subject: {email['subject'][:60]}")
    print(f"   Body: {email['body'][:200]}...")


# Cell 2: Entity extraction function
def extract_entities(text):
    """Extract financial entities from email text."""
    
    entities = {}
    
    # Amount
    amount_match = re.search(r'(?:Rs\.?|₹)\s*([\d,]+(?:\.\d{2})?)', text)
    if amount_match:
        entities['amount'] = amount_match.group(1).replace(',', '')
    
    # Type
    if 'debited' in text.lower():
        entities['type'] = 'debit'
    elif 'credited' in text.lower():
        entities['type'] = 'credit'
    
    # Account
    account_match = re.search(r'(?:account|A/C|a/c)\s*[:\s]?\s*(\w+)', text, re.IGNORECASE)
    if account_match:
        entities['account'] = account_match.group(1)
    
    # Date
    date_match = re.search(r'(\d{2}-\d{2}-\d{2,4})', text)
    if date_match:
        entities['date'] = date_match.group(1)
    
    # Reference
    ref_match = re.search(r'reference\s*(?:number|no\.?)?\s*(?:is)?\s*(\d+)', text, re.IGNORECASE)
    if ref_match:
        entities['reference'] = ref_match.group(1)
    
    return entities

# Create training examples from credit emails
credit_training = []

for email in credit_emails:
    entities = extract_entities(email['body'])
    
    # Must have at least amount and type
    if 'amount' in entities and 'type' in entities:
        example = {
            "prompt": f"Extract financial entities from this email:\n\nSubject: {email['subject']}\n\nBody: {email['body'][:1500]}",
            "completion": json.dumps(entities, indent=2)
        }
        credit_training.append(example)

print(f"Credit training examples created: {len(credit_training)}")

# Show sample
print(f"\n=== SAMPLE CREDIT TRAINING EXAMPLE ===")
print(f"PROMPT:\n{credit_training[0]['prompt'][:300]}...")
print(f"\nCOMPLETION:\n{credit_training[0]['completion']}")

# Cell 3: Load existing training data and combine
training_dir = PROJECT / "data/training"

# Load existing training data
existing_train = []
with open(training_dir / "train.jsonl", 'r') as f:
    for line in f:
        existing_train.append(json.loads(line))

existing_valid = []
with open(training_dir / "valid.jsonl", 'r') as f:
    for line in f:
        existing_valid.append(json.loads(line))

print(f"Existing training examples: {len(existing_train)}")
print(f"Existing validation examples: {len(existing_valid)}")

# Split credit data: 90% train, 10% validation
random.seed(42)
random.shuffle(credit_training)

split_idx = int(len(credit_training) * 0.9)
credit_train = credit_training[:split_idx]
credit_valid = credit_training[split_idx:]

print(f"\nNew credit train: {len(credit_train)}")
print(f"New credit valid: {len(credit_valid)}")

# Combine
combined_train = existing_train + credit_train
combined_valid = existing_valid + credit_valid

# Shuffle combined data
random.shuffle(combined_train)
random.shuffle(combined_valid)

print(f"\n=== COMBINED DATASET ===")
print(f"Total train: {len(combined_train)}")
print(f"Total valid: {len(combined_valid)}")

# Cell 4: Backup old data and save new combined data

backup_dir = PROJECT / "data/training_backup_v1"
backup_dir.mkdir(exist_ok=True)

shutil.copy(training_dir / "train.jsonl", backup_dir / "train.jsonl")
shutil.copy(training_dir / "valid.jsonl", backup_dir / "valid.jsonl")
print(f"✅ Backed up old data to {backup_dir}")

# Save combined data
with open(training_dir / "train.jsonl", 'w') as f:
    for example in combined_train:
        f.write(json.dumps(example) + '\n')

with open(training_dir / "valid.jsonl", 'w') as f:
    for example in combined_valid:
        f.write(json.dumps(example) + '\n')

print(f"✅ Saved new training data:")
print(f"   train.jsonl ({len(combined_train)} examples)")
print(f"   valid.jsonl ({len(combined_valid)} examples)")

# Check balance of debit vs credit
debit_count = sum(1 for e in combined_train if '"type": "debit"' in e['completion'])
credit_count = sum(1 for e in combined_train if '"type": "credit"' in e['completion'])

print(f"\n=== DATA BALANCE ===")
print(f"Debit examples: {debit_count}")
print(f"Credit examples: {credit_count}")

# Cell 5: Retrain command
print("=== RETRAIN WITH CREDIT DATA ===")
print()
print("Run this command in Terminal:")
print()

command = f"""cd {PROJECT}
source venv/bin/activate

mlx_lm.lora \\
    --model models/base/phi3-mini \\
    --data data/training \\
    --train \\
    --batch-size 1 \\
    --num-layers 8 \\
    --iters 600 \\
    --adapter-path models/adapters/finance-lora-v2
"""

print(command)
print()
print("Note: New adapter saved as 'finance-lora-v2'")
print("Iterations increased to 600 (more data)")

✅ Loaded 40,820 emails
Credit transaction emails found: 131

=== SAMPLE CREDIT EMAILS ===

1. Subject: Meesho IPO's 46.40% listing gains, DGCA summons IndiGo CEO, 
   Body: All you need to know about the day ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌ ﻿ ͏ ‌...

2. Subject: View: Account update for your HDFC Bank A/c
   Body: HDFC BANK Dear Customer, Rs. 50000.00 is successfully credited to your account **3545 by VPA subhashreebadatya250@okicici SUBHASHREE BADATYA on 16-11-25. Your UPI transaction reference number is 53204...

3. Subject: Received Cashback from PhonePe
   Body: Nov 1, 2025 Cashback from PhonePe ₹ 2.5 Txn. ID : T2511010958546075798981 Txn. status : Successful Credited to : Cashback/PhonePe Gift Card Amount: ₹ 2.5 Order ID : T2510252053346470859826 Hi Ranjit B...

4. Subject: Received Cashback from PhonePe
   Body: Nov 1, 2025 Cashback from PhonePe ₹ 1.5 Txn